In [0]:
%sql
USE uk_train_ride.train_rise;

In [0]:
%sql
SELECT 
        MONTH(Date_of_Journey) AS MONTH
         ,MONTHNAME(Date_of_Journey) AS MONTHS
        ,SUM(Price) AS Net_Revenue 
FROM railway
WHERE 
        Journey_Status != 'Cancelled'
GROUP BY 
        MONTHS
        ,MONTH
ORDER BY MONTH;

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Define consistent color scheme
COLORS = {
    'primary': '#1f77b4',
    'secondary': '#ff7f0e', 
    'tertiary': '#2ca02c',
    'quaternary': '#d62728',
    'purple': '#9467bd',
    'brown': '#8c564b'
}

# Get monthly revenue data
df_monthly = spark.sql("""
SELECT 
    MONTH(Date_of_Journey) AS MONTH,
    MONTHNAME(Date_of_Journey) AS MONTHS,
    SUM(Price) AS Net_Revenue 
FROM uk_train_ride.train_rise.railway
WHERE Journey_Status != 'Cancelled'
GROUP BY MONTHS, MONTH
ORDER BY MONTH
""").toPandas()

# Create line chart with area fill
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_monthly['MONTHS'], df_monthly['Net_Revenue'],
       marker='o', linewidth=3, markersize=12, color=COLORS['primary'], label='Net Revenue')
ax.fill_between(range(len(df_monthly)), df_monthly['Net_Revenue'], 
               alpha=0.3, color=COLORS['primary'])
ax.set_title('Monthly Net Revenue Trend', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Net Revenue (£)', fontsize=13)
ax.grid(True, alpha=0.3)

# Add value labels
for i, (month, val) in enumerate(zip(df_monthly['MONTHS'], df_monthly['Net_Revenue'])):
    ax.text(i, val + 3000, f'£{int(val):,}', 
           ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
%sql
SELECT 
         MONTH(Date_of_Journey) AS MONTH
         ,MONTHNAME(Date_of_Journey) AS MONTHS
        ,Ticket_Type
        ,SUM(Price) AS Net_Revenue 
FROM railway
WHERE 
        Journey_Status != 'Cancelled'
GROUP BY 
        Ticket_Type
        ,MONTH
        , MONTHS
ORDER BY MONTH;

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get ticket type revenue
df_ticket = spark.sql("""
SELECT 
    Ticket_Type,
    SUM(Price) AS Net_Revenue
FROM uk_train_ride.train_rise.railway
WHERE Journey_Status != 'Cancelled'
GROUP BY Ticket_Type
ORDER BY Net_Revenue DESC
""").toPandas()

# Create pie chart
fig, ax = plt.subplots(figsize=(10, 7))
colors = [COLORS['primary'], COLORS['secondary'], COLORS['tertiary']]
wedges, texts, autotexts = ax.pie(df_ticket['Net_Revenue'], 
                                   labels=df_ticket['Ticket_Type'],
                                   autopct='%1.1f%%',
                                   colors=colors,
                                   startangle=90,
                                   textprops={'fontsize': 13})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(14)

for text in texts:
    text.set_fontsize(14)
    text.set_fontweight('bold')

ax.set_title('Revenue Distribution by Ticket Type', fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
display(plt.show())

In [0]:
%sql
SELECT 
        Journey_Status
        ,SUM(Price) AS Net_Revenue 
        ,ROUND(COUNT(*)/Sum(COUNT(*)) OVER(),2) AS Percentage
FROM railway
GROUP BY 
        Journey_Status;

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get revenue by journey status
df_status = spark.sql("""
SELECT 
    Journey_Status,
    SUM(Price) AS Net_Revenue,
    COUNT(*) AS Journey_Count
FROM uk_train_ride.train_rise.railway
GROUP BY Journey_Status
ORDER BY Net_Revenue DESC
""").toPandas()

# Create bar chart
fig, ax = plt.subplots(figsize=(11, 7))
colors = [COLORS['tertiary'], COLORS['secondary'], COLORS['quaternary']]
bars = ax.bar(df_status['Journey_Status'], df_status['Net_Revenue'],
             color=colors, alpha=0.8, width=0.5)
ax.set_title('Revenue by Journey Status', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Net Revenue (£)', fontsize=13)
ax.set_xlabel('Journey Status', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, df_status['Net_Revenue']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 10000,
           f'£{int(val):,}',
           ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
%sql
SELECT 
        Journey_Status
        ,SUM(Price) AS Net_Revenue 
        ,ROUND(COUNT(*)/Sum(COUNT(*)) OVER() * 100,2) AS Percentage
FROM railway
WHERE Refund_Request = 'Yes'
GROUP BY 
        Journey_Status;

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Get refund data by status
df_refund = spark.sql("""
SELECT 
    Journey_Status,
    SUM(Price) AS Refund_Amount,
    COUNT(*) AS Refund_Count,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER() * 100, 1) AS Percentage
FROM uk_train_ride.train_rise.railway
WHERE Refund_Request = 'Yes'
GROUP BY Journey_Status
ORDER BY Refund_Amount DESC
""").toPandas()

# Create side-by-side comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Refund Analysis by Journey Status', fontsize=16, fontweight='bold')

colors = [COLORS['secondary'], COLORS['quaternary']]

# Chart 1: Refund Amount
bars1 = ax1.bar(df_refund['Journey_Status'], df_refund['Refund_Amount'],
               color=colors, alpha=0.8)
ax1.set_title('Total Refund Amount', fontsize=14, fontweight='bold')
ax1.set_ylabel('Refund Amount (£)', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, df_refund['Refund_Amount']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 500,
            f'£{int(val):,}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Chart 2: Refund Count Distribution
wedges, texts, autotexts = ax2.pie(df_refund['Refund_Count'], 
                                    labels=df_refund['Journey_Status'],
                                    autopct='%1.1f%%',
                                    colors=colors,
                                    startangle=90,
                                    textprops={'fontsize': 11})
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title('Refund Request Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
display(plt.show())

In [0]:
%sql
SELECT 
        Departure_Station
        ,SUM(Price) AS Net_Revenue 
        ,ROUND(COUNT(*)/Sum(COUNT(*)) OVER() * 100,2) AS Percentage
FROM railway
WHERE Refund_Request = 'Yes'
GROUP BY 
        Departure_Station
ORDER BY Percentage DESC
LIMIT 5;

In [0]:
%sql
SELECT 
        Departure_Station
        ,SUM(Price) AS Net_Revenue 
        ,ROUND(COUNT(*)/Sum(COUNT(*)) OVER() * 100,2) AS Percentage
FROM railway
WHERE Refund_Request = 'Yes'
GROUP BY 
        Departure_Station
ORDER BY Percentage ASC
LIMIT 5;